# 03 — Policy Experiments: A Small CGE Laboratory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/03_policy_experiments.ipynb)

Here we keep one calibrated baseline and compare several counterfactuals:

1. abolish all import tariffs;
2. abolish all production taxes;
3. increase the capital endowment by 10%.

## 1. Setup and calibrate once

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORKSPACE = Path("/content") if Path("/content").exists() else Path.home() / ".cache"
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKSPACE / "CGE-core-colab"

if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main", "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/miraflor/CGE-core.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core
print("✓ CGE-Core", cge_core.__version__)
print("✓ Repository:", REPO_DIR)


subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "amplpy.modules", "install", "coin"],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

module_path = subprocess.check_output(
    [sys.executable, "-m", "amplpy.modules", "path"],
    text=True,
).strip()
os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found after installing the COIN module."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pyomo.environ import prod, value

from cge_core import PyCGE, example_data
from cge_core.examples.stdcge_model_def import StdModelDef

cge = PyCGE(StdModelDef())
cge.model_data(example_data("stdcge"))
cge.model_instance("pf", "LAB")
cge.model_drop_redundant("eqpf", "LAB")
cge.model_calibrate(SOLVER)

print("✓ One baseline calibrated.")

## 2. Define a reusable scenario runner

In [ ]:
def equivalent_variation(cge):
    denom = prod(
        value(cge.base.alpha[i]) ** value(cge.base.alpha[i])
        for i in cge.base.i
    )
    return value(cge.sim.obj) / denom - value(cge.base.obj) / denom

def run_scenario(label, shocks):
    # shocks is a list of (component_name, index, new_value)
    cge.model_sim()
    for component, index, new_value in shocks:
        cge.model_modify_sim(component, index, new_value)
    cge.model_solve(SOLVER)

    frame = cge.model_compare().copy()
    return {"label": label, "results": frame, "ev": equivalent_variation(cge)}

## 3. Run three qualitatively different shocks

In [ ]:
goods = list(cge.base.i)
scenarios = []

scenarios.append(run_scenario(
    "Abolish import tariffs",
    [("taum", good, 0.0) for good in goods],
))
scenarios.append(run_scenario(
    "Abolish production taxes",
    [("tauz", good, 0.0) for good in goods],
))

capital0 = value(cge.base.FF["CAP"])
scenarios.append(run_scenario(
    "Capital endowment +10%",
    [("FF", "CAP", 1.10 * capital0)],
))

print("✓", len(scenarios), "scenarios solved")

## 4. Compare welfare

In [ ]:
welfare = pd.DataFrame(
    [{"scenario": s["label"], "equivalent_variation": s["ev"]} for s in scenarios]
)
display(welfare.style.format({"equivalent_variation": "{:+.4f}"}))

## 5. Compare output responses

In [ ]:
rows = []
for scenario in scenarios:
    part = scenario["results"]
    part = part[part["component"] == "Z"]
    for _, row in part.iterrows():
        rows.append({
            "scenario": scenario["label"],
            "good": row["index_1"],
            "pct_change": row["pct_change"],
        })

output_changes = pd.DataFrame(rows)
pivot = output_changes.pivot(index="good", columns="scenario", values="pct_change")
display(pivot)

ax = pivot.plot(kind="bar", figsize=(8, 4))
ax.axhline(0, linewidth=0.8)
ax.set_ylabel("% change from baseline")
ax.set_title("Gross output across policy scenarios")
plt.show()

## 6. Compare imports, household demand, and prices

In [ ]:
for component, title in [
    ("M", "Imports"),
    ("Xp", "Household demand"),
    ("pq", "Composite prices"),
]:
    rows = []
    for scenario in scenarios:
        part = scenario["results"]
        part = part[part["component"] == component]
        for _, row in part.iterrows():
            rows.append({
                "scenario": scenario["label"],
                "item": row["index_1"],
                "pct_change": row["pct_change"],
            })
    table = pd.DataFrame(rows).pivot(index="item", columns="scenario", values="pct_change")
    print(title)
    display(table.style.format("{:+.2f}%"))

## 7. Make your own scenario 👇

A shock is a tuple `(component_name, index, new_value)`. Combine tuples for a policy package.

In [ ]:
# 👇 EDIT THIS POLICY PACKAGE
CUSTOM_SHOCKS = [
    ("taum", "BRD", 0.05),
]

custom = run_scenario("My custom scenario", CUSTOM_SHOCKS)

display(
    custom["results"][
        custom["results"]["component"].isin(["Z", "M", "Xp", "pq"])
    ][["component", "index_1", "base_value", "sim_value", "pct_change"]]
    .style.format({
        "base_value": "{:.4f}",
        "sim_value": "{:.4f}",
        "pct_change": "{:+.2f}%",
    })
)
print(f"Equivalent variation: {custom['ev']:+.4f}")

## What you learned

A **model** is the economic structure. A **scenario** specifies exogenous changes. A **closure** specifies which variables are fixed and which absorb adjustment.

## Next

Notebook 04 turns a single SAM CSV into model-ready CGE-Core data.

[Open Notebook 04 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/04_bring_your_own_sam.ipynb)